# Project 2

## Imports

In [1]:
!pip install spacy torch textworld[gym] tqdm scikit-learn && python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 16.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.5/101.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 57.7 MB/s eta 0:00:00
  Created wheel for jericho: filename=jericho-3.3.0-py3-none-any.whl size=325237 sha256=9376bf722f694b9e53e9523aa2a4655c27c4c4ba1b44362e4a77216e79519b8b
  Stored in directory: /root/.cache/pip/wheels/a5/fa/71/4ccd66162c5fce936673106cf00a3b66b6937db5dfd0dac581
Successfully built jericho
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 145.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
import os
import json
import random
import re

import spacy
import torch
import textworld
import textworld.gym

from collections import Counter
from tqdm import tqdm

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

## Part 1


In [5]:
def parse_verb_object(s):
    return s.split()[0], ' '.join(s.split()[1:])

def walkthrough_and_record(game_dir, max_games=500):

    triples = []
    seen = 0

    files = list(os.listdir(game_dir))

    # let's make it reproducible
    random.seed(42)
    random.shuffle(files)

    for file in files:

        # jsons have the walkthroughs... if no json, no need to look at this game
        if not file.endswith('.json'): continue

        # cap the total games for speed
        seen += 1
        if seen > max_games: break

        # extract walkthrough from the json meta data
        # walkthrough == winning actions
        x = json.load(open(os.path.join(game_dir, file), 'r'))
        wt = x['extras']['walkthrough']

        # setup the actual game we can interact with to get observations
        ulx = os.path.join(game_dir, file.replace('.json', '.ulx'))
        game_id = textworld.gym.register_game(ulx)
        env = textworld.gym.make(game_id)

        # record observations and track the moves we made at each to win
        obs, *_ = env.reset()

        for move in wt:
            triples.append((obs, *parse_verb_object(move)))
            obs, *_ = env.step(move)

        # don't forget the last move we just made
        triples.append((obs, *parse_verb_object(move)))

    return triples

gdir = 'cog2019_ftwp/games/train'

# triples of obs, verb, obj
print('Extracting data for ML... this could take a while')
ml_data = walkthrough_and_record(gdir)
print('> We extracted', len(ml_data), '(obs, act) pairs.')

Extracting data for ML... this could take a while


FileNotFoundError: [Errno 2] No such file or directory: 'cog2019_ftwp/games/train'

## Part 2